# Лабораторная 2 (GPU-вариант): декодерный трансформер на `Tiny Shakespeare`

Цель работы: выполнить расширенный перенос на реальный корпус в режиме `GPU-friendly`,
увеличить масштаб модели и подтвердить улучшение относительно `CPU`-варианта.

## Что нужно знать до старта
- Полностью завершён `CPU`-вариант `ЛР02` с зафиксированным `test_perplexity`.
- GPU-среда проходит `gpu_preflight()` без fallback и нестабильных kernels.
- Контракт gate-критериев остаётся тем же: сравнение с baseline + generation checks.

## Интуиция задачи без формул
GPU-вариант не меняет математику задачи, а масштабирует обучение в пределах зафиксированного бюджета.
Если `gpu_preflight`, causal-mask и leakage checks выполнены, улучшение качества объясняется обучением модели, а не сменой условий эксперимента.

## Контракт данных (быстрый ориентир)
- `token_windows` и `targets`: формы `(batch, context)` как в CPU-треке.
- `padding_mask`: исключает `PAD` из loss/metric.
- `causal_mask`: строго нижнетреугольная, без доступа к будущим позициям.
- `gpu_preflight`: обязательный preflight до длинного обучения, без скрытого CPU fallback.
- Итоговые проверки: baseline/perplexity/generation/leakage + индикатор `CPU_REFERENCE_PERPLEXITY`.



## Маршрут выполнения

Строгий порядок шагов:
1. Завершить `CPU`-вариант `ЛР02` и зафиксировать `test_perplexity`.
2. `TODO 1`: загрузка корпуса и построение словаря.
3. `TODO 2`: окна фиксированной длины и индексное разбиение.
4. `TODO 3`: декодерный блок с причинной маской.
5. `TODO 4`: обучение, метрики и сравнение с частотным ориентиром.
6. `TODO 5`: детерминированная генерация по фиксированным подсказкам.
7. `TODO 6`: диагностика внимания без доступа в будущее.


In [ ]:
import ctypes
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

RUNTIME_MODE = os.environ.get("COURSE_RUNTIME_MODE", "local-gpu")
COURSE_REPO_HTTPS_URL = os.environ.get(
    "COURSE_REPO_HTTPS_URL",
    "https://github.com/<org>/<repo>.git",
)
NOTEBOOK_REQUIREMENTS = "themes/04-Autoregression/lab/requirements.txt"


def _prepend_path_env(var_name: str, new_paths: list[Path]) -> None:
    """Добавляет пути в начало переменной окружения с путями.

    Аргументы:
      var_name: Имя переменной окружения (например, `LD_LIBRARY_PATH`).
      new_paths: Пути-кандидаты, которые нужно добавить в начало.

    Возвращает:
      `None`.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    existing = os.environ.get(var_name, "")
    cleaned_new = []
    for path in new_paths:
        if path.is_dir():
            cleaned_new.append(str(path))

    if not cleaned_new:
        return

    existing_parts = [part for part in existing.split(":") if part]
    merged = []
    for part in cleaned_new + existing_parts:
        if part not in merged:
            merged.append(part)
    os.environ[var_name] = ":".join(merged)


def _detect_site_packages_dir() -> Path | None:
    """Находит каталог `site-packages` активной виртуальной среды.

    Аргументы:
      Нет.

    Возвращает:
      Путь к `site-packages` или `None`, если каталог не найден.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    major, minor = sys.version_info[:2]
    candidate = Path(sys.prefix) / "lib" / f"python{major}.{minor}" / "site-packages"
    if candidate.is_dir():
        return candidate
    return None


def _preload_cuda_runtime_libraries(site_packages: Path) -> dict:
    """Предзагружает CUDA-библиотеки в текущий процесс до импорта TensorFlow.

    Аргументы:
      site_packages: Каталог `site-packages` активной виртуальной среды.

    Возвращает:
      Сводка со списками успешно загруженных, отсутствующих и проблемных библиотек.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    nvidia_root = site_packages / "nvidia"
    library_specs = [
        ("cuda_runtime", "libcudart.so.12"),
        ("cublas", "libcublas.so.12"),
        ("cublas", "libcublasLt.so.12"),
        ("cudnn", "libcudnn.so.9"),
        ("cufft", "libcufft.so.11"),
        ("curand", "libcurand.so.10"),
        ("cusolver", "libcusolver.so.11"),
        ("cusparse", "libcusparse.so.12"),
        ("nccl", "libnccl.so.2"),
        ("nvjitlink", "libnvJitLink.so.12"),
    ]

    loaded = []
    missing = []
    failed = []

    for subdir, library_name in library_specs:
        library_path = nvidia_root / subdir / "lib" / library_name
        if not library_path.is_file():
            missing.append(str(library_path))
            continue
        try:
            ctypes.CDLL(str(library_path), mode=ctypes.RTLD_GLOBAL)
            loaded.append(str(library_path))
        except OSError as exc:
            failed.append(f"{library_path}: {exc}")

    return {
        "loaded": loaded,
        "missing": missing,
        "failed": failed,
    }


def _configure_local_gpu_runtime_env(runtime_mode: str) -> dict:
    """Готовит переменные окружения для локального запуска TensorFlow на GPU.

    Аргументы:
      runtime_mode: Запрошенный режим выполнения тетради.

    Возвращает:
      Словарь с краткой сводкой применённой настройки путей.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    if runtime_mode != "local-gpu":
        return {
            "applied": False,
            "reason": "runtime_mode != local-gpu",
        }

    site_packages = _detect_site_packages_dir()
    if site_packages is None:
        return {
            "applied": False,
            "reason": "site-packages not found",
        }

    tensorflow_dir = site_packages / "tensorflow"
    nvidia_root = site_packages / "nvidia"
    nvidia_lib_dirs = sorted(path for path in nvidia_root.glob("*/lib") if path.is_dir())
    _prepend_path_env("LD_LIBRARY_PATH", [tensorflow_dir, *nvidia_lib_dirs])

    nvcc_bin = nvidia_root / "cuda_nvcc" / "bin"
    _prepend_path_env("PATH", [nvcc_bin])

    preload_report = _preload_cuda_runtime_libraries(site_packages)

    return {
        "applied": True,
        "tensorflow_dir": str(tensorflow_dir),
        "nvidia_lib_dirs": [str(path) for path in nvidia_lib_dirs],
        "nvcc_bin": str(nvcc_bin),
        "preload_report": preload_report,
    }


gpu_env_info = _configure_local_gpu_runtime_env(RUNTIME_MODE)
print("gpu_env_info:", gpu_env_info)


def _detect_notebook_platform():
    """Определяет тип среды выполнения текущей тетради.

    Аргументы:
      Нет.

    Возвращает:
      Строка из множества `{'local', 'colab', 'kaggle'}`.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"


def _looks_like_repo_root(path: Path) -> bool:
    """Проверяет, похож ли путь на корень учебного репозитория.

    Аргументы:
      path: Проверяемый путь.

    Возвращает:
      `True`, если обнаружены ключевые признаки корня репозитория.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return (
        path.is_dir()
        and (path / "themes").is_dir()
        and (path / "course_runtime.py").is_file()
    )


def _canonical_cloud_repo_root(platform: str) -> Path:
    """Возвращает стандартный путь клонирования для облачной платформы.

    Аргументы:
      platform: Имя платформы (`'colab'` или `'kaggle'`).

    Возвращает:
      Абсолютный путь каталога репозитория.

    Исключения:
      ValueError: Если передано неподдерживаемое имя платформы.
    """
    if platform == "colab":
        return Path("/content/students-AI_math_essentials")
    if platform == "kaggle":
        return Path("/kaggle/working/students-AI_math_essentials")
    raise ValueError(f"Unexpected cloud platform: {platform}")


def _is_placeholder_repo_url(repo_url: str) -> bool:
    """Проверяет, остался ли в настройке шаблонный URL репозитория.

    Аргументы:
      repo_url: Проверяемый URL репозитория.

    Возвращает:
      `True`, если URL имеет вид шаблона-заглушки.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return repo_url.strip() == "https://github.com/<org>/<repo>.git"


def _find_repo_root_from_cwd():
    """Ищет корень курса, поднимаясь от текущего каталога вверх.

    Аргументы:
      Нет.

    Возвращает:
      Объект `Path` корня репозитория или `None`, если путь не найден.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if _looks_like_repo_root(candidate):
            return candidate
    return None


def _ensure_course_runtime_importable(runtime_mode: str, repo_url: str) -> None:
    """Обеспечивает доступность модуля `course_runtime` для текущей среды.

    Аргументы:
      runtime_mode: Режим запуска тетради.
      repo_url: URL репозитория курса для облачной автозагрузки.

    Возвращает:
      `None`.

    Исключения:
      ModuleNotFoundError: Если локальный запуск выполнен вне корректного корня репозитория.
      RuntimeError: Если в облаке отсутствует валидный URL репозитория или каталог повреждён.
      subprocess.CalledProcessError: Если команда `git clone` завершается с ошибкой.
    """
    if importlib.util.find_spec("course_runtime") is not None:
        return

    local_repo_root = _find_repo_root_from_cwd()
    if local_repo_root is not None:
        if str(local_repo_root) not in sys.path:
            sys.path.insert(0, str(local_repo_root))
        return

    platform = _detect_notebook_platform()
    if platform == "local":
        raise ModuleNotFoundError(
            "Не удалось импортировать course_runtime.py. Для локального запуска "
            "открывайте репозиторий через `.venv/bin/jupyter notebook` из корня проекта."
        )

    repo_root = _canonical_cloud_repo_root(platform)
    if not _looks_like_repo_root(repo_root):
        if _is_placeholder_repo_url(repo_url):
            raise RuntimeError(
                "Для cloud auto-bootstrap нужен публичный HTTPS URL курса. "
                "Замените COURSE_REPO_HTTPS_URL на реальный адрес репозитория."
            )
        repo_root.parent.mkdir(parents=True, exist_ok=True)
        if repo_root.exists() and any(repo_root.iterdir()):
            raise RuntimeError(
                f"Каталог {repo_root} уже существует, но не выглядит как корень курса. "
                "Очистите runtime или используйте новый notebook session."
            )
        print(f"Bootstrapping course repository into {repo_root} ...")
        subprocess.run(["git", "clone", repo_url, str(repo_root)], check=True)

    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


_ensure_course_runtime_importable(RUNTIME_MODE, COURSE_REPO_HTTPS_URL)

from course_runtime import setup_notebook_runtime

runtime_info = setup_notebook_runtime(
    runtime_mode=RUNTIME_MODE,
    course_repo_https_url=COURSE_REPO_HTTPS_URL,
    notebook_requirements=NOTEBOOK_REQUIREMENTS,
)
runtime_info.as_dict()


## Константы GPU-варианта

Этот трек выполняется только при наличии графического процессора.
Если `GPU` недоступен, работу по этой тетради нужно перенести в среду с корректно установленными CUDA-драйверами.

Перед запуском установите `COURSE_RUNTIME_MODE=local-gpu`.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path

SEED = 29
PAD_ID = 0
CHECK_GEN_STEPS = 16
PROMPT_COUNT = 20
CPU_REFERENCE_PERPLEXITY = 7.64

GPU_60M_PROFILE = {
    'chars': 900_000,
    'context': 128,
    'stride': 1,
    'batch_size': 128,
    'embed_dim': 192,
    'num_heads': 6,
    'ff_dim': 384,
    'eval_every_steps': 1200,
    'validation_steps': 64,
    'max_eval_rounds': 180,
    'early_stopping_patience': 16,
    'early_stopping_min_delta': 1e-4,
    'early_stopping_probe_floor': 18,
    'min_minutes_before_early_stop': 45,
    'max_training_minutes': 60,
    'min_minutes_before_generation_stop': 20,
    'generation_stop_patience': 2,
    'generation_probe_every': 4,
    'learning_rate': 1e-3,
    'gen_match_ratio': 0.18,
    'gen_threshold': 19,
    'gen_mean_threshold': 0.25,
}

GPU_60M_PROFILE_BOOST = {
    **GPU_60M_PROFILE,
    'eval_every_steps': 1500,
    'validation_steps': 64,
    'generation_probe_every': 3,
    'early_stopping_patience': 20,
    'learning_rate': 8e-4,
}

GPU_PROFILE_MAP = {
    'gpu_60m': GPU_60M_PROFILE,
    'gpu_60m_boost': GPU_60M_PROFILE_BOOST,
}


def gpu_preflight(runtime_info):
    """Проверяет готовность локального GPU-контура перед длинным обучением.

    Аргументы:
      runtime_info: Результат `setup_notebook_runtime(...)` с полем `effective_mode`.

    Возвращает:
      Словарь с краткой сводкой по обнаруженным GPU и версии TensorFlow.

    Исключения:
      RuntimeError: Если режим запуска не `local-gpu`, GPU не виден,
        либо GPU-ядра нестабильны на реальной короткой нагрузке.
    """
    effective_mode = getattr(runtime_info, 'effective_mode', '')
    print('effective_mode:', effective_mode)
    if effective_mode != 'local-gpu':
        raise RuntimeError(
            'GPU-вариант ЛР02 должен запускаться только в режиме local-gpu. '
            'Установите COURSE_RUNTIME_MODE=local-gpu и перезапустите ядро.'
        )

    try:
        nvidia_report = subprocess.run(
            ['nvidia-smi', '-L'],
            check=True,
            capture_output=True,
            text=True,
        )
        lines = [line for line in nvidia_report.stdout.strip().splitlines() if line.strip()]
        print('nvidia-smi -L (первые строки):')
        for line in lines[:3]:
            print('  ', line)
    except Exception as exc:
        raise RuntimeError(
            'GPU не виден (setup): команда nvidia-smi недоступна или вернула ошибку. '
            'Используйте уже подготовленную среду из '
            'themes/00-Foundations/guides/05_local_tensorflow_gpu_notebooks.md '
            'и не переустанавливайте ОС внутри ЛР.'
        ) from exc

    print('TensorFlow version:', tf.__version__)
    print('TensorFlow built with CUDA:', tf.test.is_built_with_cuda())
    build_info = tf.sysconfig.get_build_info()
    print('TensorFlow build cuda_version:', build_info.get('cuda_version', 'unknown'))
    print('TensorFlow build cudnn_version:', build_info.get('cudnn_version', 'unknown'))

    physical_gpus = tf.config.list_physical_devices('GPU')
    logical_gpus = tf.config.list_logical_devices('GPU')
    print('Physical GPUs:', [device.name for device in physical_gpus])
    print('Logical GPUs :', [device.name for device in logical_gpus])

    if len(physical_gpus) == 0 or len(logical_gpus) == 0:
        raise RuntimeError(
            'GPU не виден (setup): TensorFlow не зарегистрировал GPU-устройства. '
            'Проверьте окружение по guide 05/06 в themes/00-Foundations.'
        )

    try:
        # Проверка 1: базовая матричная операция на /GPU:0.
        with tf.device('/GPU:0'):
            lhs = tf.random.normal((128, 128), dtype=tf.float32)
            rhs = tf.random.normal((128, 128), dtype=tf.float32)
            product = tf.matmul(lhs, rhs)
            _ = float(tf.reduce_mean(product).numpy())

        # Проверка 2: короткий notebook-like шаг обучения Keras на /GPU:0.
        features = np.random.normal(size=(32, 8)).astype('float32')
        targets = np.random.normal(size=(32, 1)).astype('float32')
        with tf.device('/GPU:0'):
            smoke_model = keras.Sequential(
                [
                    layers.Input(shape=(8,)),
                    layers.Dense(16, activation='relu'),
                    layers.Dense(1),
                ],
                name='gpu_smoke_model',
            )
            smoke_model.compile(
                optimizer=keras.optimizers.Adam(1e-3),
                loss='mse',
            )
            smoke_model.train_on_batch(features, targets)
    except Exception as exc:
        raise RuntimeError(
            'GPU виден, но kernels падают (compatibility). Это не ошибка кода ЛР. '
            'См. themes/00-Foundations/guides/07_tensorflow_blackwell_local_gpu_case_study.md. '
            f'Исходная ошибка: {type(exc).__name__}: {exc}'
        ) from exc

    print('gpu_preflight(): PASSED')
    return {
        'tensorflow_version': tf.__version__,
        'cuda_version': build_info.get('cuda_version', 'unknown'),
        'cudnn_version': build_info.get('cudnn_version', 'unknown'),
        'physical_gpus': [device.name for device in physical_gpus],
        'logical_gpus': [device.name for device in logical_gpus],
    }


RUN_MODE = 'gpu'
selected_profile_name = os.environ.get('GPU_PROFILE_NAME', 'gpu_60m').strip().lower()
if selected_profile_name not in GPU_PROFILE_MAP:
    raise ValueError(
        f"Неизвестный профиль GPU: {selected_profile_name}. "
        f"Допустимые значения: {sorted(GPU_PROFILE_MAP)}"
    )

cfg = dict(GPU_PROFILE_MAP[selected_profile_name])
cfg['max_training_minutes'] = float(
    os.environ.get('GPU_TRAINING_BUDGET_MINUTES', cfg['max_training_minutes'])
)
cfg['warmup_steps'] = int(os.environ.get('GPU_WARMUP_STEPS', '2'))
cfg['warmup_probe_steps'] = int(os.environ.get('GPU_WARMUP_PROBE_STEPS', '2'))

if cfg['max_training_minutes'] <= 0:
    raise ValueError('GPU_TRAINING_BUDGET_MINUTES должен быть положительным числом.')
if cfg['warmup_steps'] < 1:
    raise ValueError('GPU_WARMUP_STEPS должен быть целым числом >= 1.')
if cfg['warmup_probe_steps'] < 1:
    raise ValueError('GPU_WARMUP_PROBE_STEPS должен быть целым числом >= 1.')

plt.style.use('default')
keras.utils.set_random_seed(SEED)

preflight_info = gpu_preflight(runtime_info)
print('Режим выполнения:', RUN_MODE)
print('Профиль GPU:', selected_profile_name)
print('Измеряемый бюджет обучения (мин):', cfg['max_training_minutes'])


## Математический ориентир

Мы оптимизируем токенную перекрёстную энтропию (cross-entropy), которая эквивалентна
среднему отрицательному лог-правдоподобию для next-token задачи.

Перплексия (perplexity) вычисляется как:

$$
\mathrm{PPL} = e^{\mathrm{loss}}
$$

Для режима `gpu` дополнительно проверяется улучшение относительно
ориентира `CPU_REFERENCE_PERPLEXITY`.


## TODO 1: загрузка корпуса и построение словаря


In [ ]:
def load_tiny_shakespeare(profile_cfg):
    """Загружает корпус и строит детерминированное символьное кодирование.

    Аргументы:
      profile_cfg: Словарь параметров выбранного профиля.

    Возвращает:
      Кортеж `(text, encoded_ids, vocab, char_to_id, id_to_char)`.

    Исключения:
      ValueError: Если выбранный профиль задаёт слишком короткий фрагмент текста.
    """
    # TODO 1.1: Загрузите корпус Tiny Shakespeare через tf.keras.utils.get_file.
    # Скачиваем файл с корпусом Шекспира
    path_to_file = tf.keras.utils.get_file(
        'shakespeare.txt',
        'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
    )
    
    # Читаем текст
    with open(path_to_file, 'r', encoding='utf-8') as f:
        text = f.read()
    
    # TODO 1.2: Возьмите детерминированный срез длины profile_cfg['chars'].
    chars_to_take = profile_cfg['chars']
    if chars_to_take > len(text):
        raise ValueError(f'Профиль требует {chars_to_take} символов, но корпус содержит только {len(text)} символов.')
    
    text = text[:chars_to_take]
    
    # TODO 1.3: Постройте словарь ['<PAD>', ...] и кодирование в int32.
    # Получаем уникальные символы в тексте
    unique_chars = sorted(set(text))
    
    # Создаем словарь с PAD_ID на позиции 0
    vocab = ['<PAD>'] + unique_chars
    
    # Создаем отображения символ -> ID и ID -> символ
    char_to_id = {ch: idx for idx, ch in enumerate(vocab)}
    id_to_char = {idx: ch for ch, idx in char_to_id.items()}
    
    # Кодируем текст в ID
    encoded_ids = np.array([char_to_id[ch] for ch in text], dtype=np.int32)
    
    return text, encoded_ids, vocab, char_to_id, id_to_char


# TODO 1.4: Вызовите load_tiny_shakespeare(cfg) и сохраните результаты в
# text, encoded_ids, vocab, char_to_id, id_to_char.
text, encoded_ids, vocab, char_to_id, id_to_char = load_tiny_shakespeare(cfg)

# Выводим информацию о загруженных данных
print(f"Корпус загружен успешно!")
print(f"Длина текста: {len(text)} символов")
print(f"Размер словаря (включая <PAD>): {len(vocab)}")
print(f"Первые 100 символов текста:\n{text[:100]}")
print(f"\nПервые 20 ID:\n{encoded_ids[:20]}")
print(f"\nСловарь (первые 10 символов):")
for i, (char, idx) in enumerate(list(char_to_id.items())[:10]):
    print(f"  {idx}: '{char}'")
print(f"  ...")
print(f"\nПример декодирования первых 10 ID: {''.join([id_to_char[id] for id in encoded_ids[:10]])}")

# Дополнительная проверка: убеждаемся что PAD_ID = 0
assert char_to_id['<PAD>'] == 0, "PAD_ID должен быть 0"
print(f"\n✓ PAD_ID = {PAD_ID}")

# Проверка детерминизма при одинаковых параметрах
print("\nПроверка детерминизма...")
text2, ids2, vocab2, _, _ = load_tiny_shakespeare(cfg)
assert np.array_equal(encoded_ids, ids2), "Данные не детерминированы!"
assert text == text2, "Текст не детерминирован!"
print("✓ Данные детерминированы при одинаковых параметрах")

In [ ]:
# Мини-проверка после TODO 1
assert len(text) == cfg['chars'], 'Длина среза текста не совпадает с профилем.'
assert encoded_ids.dtype == np.int32, 'Кодирование должно быть int32.'
assert vocab[PAD_ID] == '<PAD>', 'Нулевой идентификатор должен быть зарезервирован под PAD.'

text_b, ids_b, _, _, _ = load_tiny_shakespeare(cfg)
assert np.array_equal(encoded_ids[:1000], ids_b[:1000]), 'Кодирование должно быть воспроизводимым.'

print('Режим выполнения:', RUN_MODE)
print('Длина текста:', len(text))
print('Размер словаря:', len(vocab))


## TODO 2: окна фиксированной длины и разбиение


In [ ]:
def build_windows(encoded_stream, context_len, stride):
    """Строит обучающие окна фиксированной длины.

    Аргументы:
      encoded_stream: Одномерный массив кодов символов.
      context_len: Длина входного контекста.
      stride: Шаг между соседними окнами.

    Возвращает:
      Кортеж `(X, Y, starts)`.

    Исключения:
      ValueError: Если поток слишком короткий для построения хотя бы одного окна.
    """
    # TODO 2.1: Постройте X и Y со сдвигом на один шаг.
    stream_len = len(encoded_stream)
    
    # Минимальная длина для одного окна: context_len + 1 (для target)
    if stream_len < context_len + 1:
        raise ValueError(f'Поток слишком короткий: {stream_len} < {context_len + 1}')
    
    # Вычисляем количество окон
    num_windows = (stream_len - context_len) // stride + 1
    
    # Инициализируем массивы X и Y
    X = np.zeros((num_windows, context_len), dtype=np.int32)
    Y = np.zeros((num_windows, context_len), dtype=np.int32)
    starts = []
    
    # Строим окна с заданным шагом
    for i in range(num_windows):
        start_idx = i * stride
        end_idx = start_idx + context_len
        
        # X: контекст длины context_len
        X[i] = encoded_stream[start_idx:end_idx]
        
        # Y: следующий токен для каждой позиции контекста
        Y[i] = encoded_stream[start_idx + 1:end_idx + 1]
        
        starts.append(start_idx)
    
    # TODO 2.2: Верните также массив стартовых индексов starts.
    return X, Y, np.array(starts, dtype=np.int32)


# TODO 2.3: Выполните индексное разбиение X_all, y_all, starts_all на
# train/val/test без случайного перемешивания.

# Параметры для построения окон
CONTEXT_LEN = cfg['context']  # 128 для GPU профиля
STRIDE = cfg['stride']        # 1 для GPU профиля

# Строим все окна
X_all, y_all, starts_all = build_windows(encoded_ids, CONTEXT_LEN, STRIDE)

print(f"Всего окон: {len(X_all)}")
print(f"X_all shape: {X_all.shape}")
print(f"y_all shape: {y_all.shape}")
print(f"starts_all shape: {starts_all.shape}")
print(f"Первые 5 стартовых индексов: {starts_all[:5]}")
print(f"Последние 5 стартовых индексов: {starts_all[-5:]}")

# Детерминированное разбиение без перемешивания (80/10/10)
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

total_windows = len(X_all)
train_end = int(total_windows * train_ratio)
val_end = train_end + int(total_windows * val_ratio)

# Разбиваем по порядку (без shuffle)
X_train = X_all[:train_end]
y_train = y_all[:train_end]
starts_train = starts_all[:train_end]

X_val = X_all[train_end:val_end]
y_val = y_all[train_end:val_end]
starts_val = starts_all[train_end:val_end]

X_test = X_all[val_end:]
y_test = y_all[val_end:]
starts_test = starts_all[val_end:]

print(f"\nРазмеры выборок:")
print(f"  Train: {len(X_train)} окон ({len(X_train)/total_windows*100:.1f}%)")
print(f"  Val:   {len(X_val)} окон ({len(X_val)/total_windows*100:.1f}%)")
print(f"  Test:  {len(X_test)} окон ({len(X_test)/total_windows*100:.1f}%)")

# Проверяем, что разбиение детерминированное
print(f"\nПроверка формы:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  X_val shape: {X_val.shape}")
print(f"  X_test shape: {X_test.shape}")

# Визуализация примера окна
sample_idx = 0
sample_X = X_train[sample_idx]
sample_y = y_train[sample_idx]

print(f"\nПример окна (индекс {sample_idx}):")
print(f"  X (контекст, {CONTEXT_LEN} символов):")
print(f"    {''.join([id_to_char[id] for id in sample_X[:50]])}...")
print(f"  Y (цели, сдвинуты на 1):")
print(f"    {''.join([id_to_char[id] for id in sample_y[:50]])}...")

# Проверяем корректность сдвига
assert np.all(sample_X[1:] == sample_y[:-1]), "Ошибка: сдвиг X и Y не совпадает!"
print(f"\n✓ Проверка сдвига: X[1:] == Y[:-1]")

# Сохраняем информацию о разбиении для дальнейшего использования
split_info = {
    'train': {'X': X_train, 'y': y_train, 'starts': starts_train},
    'val': {'X': X_val, 'y': y_val, 'starts': starts_val},
    'test': {'X': X_test, 'y': y_test, 'starts': starts_test}
}

In [ ]:
# Мини-проверка после TODO 2
assert X_train.shape[1] == cfg['context']
assert y_train.shape == X_train.shape
assert starts_train.ndim == 1

# Проверка сдвига внутри окон.
assert np.array_equal(X_train[0, 1:], y_train[0, :-1]), 'Сдвиг X/Y нарушен.'

print('Окон train/val/test:', len(X_train), len(X_val), len(X_test))


## TODO 3: декодерный блок с причинной маской


In [ ]:
def build_causal_mask(seq_len):
    """Строит нижнетреугольную причинную маску.

    Аргументы:
      seq_len: Длина последовательности.

    Возвращает:
      Булев тензор формы `(seq_len, seq_len)`.

    Исключения:
      tf.errors.InvalidArgumentError: Если `seq_len` не является положительным.
    """
    # Проверяем что seq_len положительный
    tf.debugging.assert_positive(seq_len)
    
    # Создаем нижнетреугольную маску (включая диагональ)
    mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
    
    # Преобразуем в булевый тензор
    mask = tf.cast(mask, tf.bool)
    
    return mask


class TokenAndPositionEmbedding(layers.Layer):
    """Складывает токенные и позиционные векторы."""

    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        """Инициализирует слой токенного и позиционного встраивания."""
        super().__init__(**kwargs)
        if embed_dim < 1:
            raise ValueError('embed_dim должен быть положительным.')
        self.token_embedding = layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.position_embedding = layers.Embedding(maxlen, embed_dim)

    def call(self, inputs):
        """Суммирует токенные и позиционные векторы."""
        # Получаем длину последовательности
        seq_len = tf.shape(inputs)[1]
        
        # Создаем позиции от 0 до seq_len-1
        positions = tf.range(start=0, limit=seq_len, delta=1)
        
        # Получаем эмбеддинги
        token_embeds = self.token_embedding(inputs)
        position_embeds = self.position_embedding(positions)
        
        # Складываем
        return token_embeds + position_embeds

    def compute_mask(self, inputs, mask=None):
        """Пробрасывает маску непустых токенов."""
        return self.token_embedding.compute_mask(inputs)


class CausalDecoderBlock(layers.Layer):
    """Минимальный декодерный блок с причинной маской."""

    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        """Создаёт внутренние слои декодерного блока."""
        super().__init__(**kwargs)
        if embed_dim % num_heads != 0:
            raise ValueError('embed_dim должен делиться на num_heads без остатка.')
        self.self_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=rate,
        )
        self.ffn = keras.Sequential(
            [
                layers.Dense(ff_dim, activation='relu'),
                layers.Dense(embed_dim),
            ]
        )
        self.norm_1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm_2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout_1 = layers.Dropout(rate)
        self.dropout_2 = layers.Dropout(rate)

    def call(self, inputs, padding_mask=None, training=None, return_attention_scores=False):
        """Прогоняет вход через маскированное самовнимание и FFN."""
        seq_len = tf.shape(inputs)[1]
        
        # Создаем причинную маску
        causal_mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        causal_mask = tf.cast(causal_mask, tf.bool)
        
        # Комбинируем с padding_mask если необходимо
        if padding_mask is not None:
            # padding_mask: (batch, seq_len) -> (batch, 1, 1, seq_len)
            padding_mask = padding_mask[:, tf.newaxis, tf.newaxis, :]
            # Объединяем маски
            combined_mask = causal_mask[tf.newaxis, tf.newaxis, :, :] & padding_mask
        else:
            combined_mask = causal_mask[tf.newaxis, tf.newaxis, :, :]
        
        # Self-attention с маской
        attn_output, attn_scores = self.self_attention(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=combined_mask,
            return_attention_scores=True,
            training=training,
        )
        
        # Residual + Normalization
        attn_output = self.dropout_1(attn_output, training=training)
        out_1 = self.norm_1(inputs + attn_output)
        
        # FFN
        ffn_output = self.ffn(out_1, training=training)
        ffn_output = self.dropout_2(ffn_output, training=training)
        
        # Residual + Normalization
        out_2 = self.norm_2(out_1 + ffn_output)
        
        if return_attention_scores:
            return out_2, attn_scores
        return out_2

    def compute_mask(self, inputs, mask=None):
        """Отключает автоматическое пробрасывание маски в выход блока."""
        return None


def masked_sparse_crossentropy(y_true, y_pred):
    """Считает перекрёстную энтропию по непустым позициям."""
    per_token = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, PAD_ID), tf.float32)
    masked_loss = per_token * mask
    return tf.reduce_sum(masked_loss) / (tf.reduce_sum(mask) + 1e-8)


def masked_token_accuracy(y_true, y_pred):
    """Считает токенную точность по непустым позициям."""
    pred = tf.argmax(y_pred, axis=-1, output_type=y_true.dtype)
    correct = tf.cast(tf.equal(y_true, pred), tf.float32)
    mask = tf.cast(tf.not_equal(y_true, PAD_ID), tf.float32)
    return tf.reduce_sum(correct * mask) / (tf.reduce_sum(mask) + 1e-8)


def perplexity_from_loss(loss_value):
    """Преобразует значение функции потерь в перплексию."""
    return float(np.exp(loss_value))


def frequency_baseline_perplexity(y_train_data, y_test_data, vocab_size, pad_id=PAD_ID):
    """Считает частотный базовый ориентир перплексии."""
    train_tokens = y_train_data[y_train_data != pad_id].reshape(-1)
    test_tokens = y_test_data[y_test_data != pad_id].reshape(-1)
    if len(train_tokens) == 0 or len(test_tokens) == 0:
        raise ValueError('Для базового ориентира нужны непустые токены.')

    counts = np.bincount(train_tokens, minlength=vocab_size).astype(np.float64)
    probs = counts / counts.sum()
    probs = np.maximum(probs, 1e-12)

    nll = -np.mean(np.log(probs[test_tokens]))
    return float(np.exp(nll))

## TODO 4: сборка и обучение модели

Обязательный контракт GPU-варианта:
- лимит времени `cfg['max_training_minutes']` минут (по умолчанию `90`);
- валидация каждые `cfg['eval_every_steps']` шагов;
- ранняя остановка по `val_loss`;
- целевая генерация: не ниже `19/20` фиксированных подсказок.


In [ ]:
# TODO 4.1: Соберите модель decoder-only с TokenAndPositionEmbedding и CausalDecoderBlock.
# TODO 4.2: Скомпилируйте модель с masked_sparse_crossentropy и masked_token_accuracy.

# Параметры модели
VOCAB_SIZE = len(vocab)
EMBED_DIM = cfg['embed_dim']  # 192
NUM_HEADS = cfg['num_heads']  # 6
FF_DIM = cfg['ff_dim']        # 384
NUM_LAYERS = 3
DROPOUT_RATE = 0.1
LEARNING_RATE = cfg['learning_rate']  # 1e-3

# Строим модель
inputs = layers.Input(shape=(CONTEXT_LEN,), name='input_tokens')

# Embedding слой
x = TokenAndPositionEmbedding(
    maxlen=CONTEXT_LEN,
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    name='token_position_embedding'
)(inputs)

# Добавляем несколько декодерных блоков
for i in range(NUM_LAYERS):
    x = CausalDecoderBlock(
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        ff_dim=FF_DIM,
        rate=DROPOUT_RATE,
        name=f'causal_decoder_block_{i}'
    )(x, training=True)

# Выходной слой
outputs = layers.Dense(VOCAB_SIZE, activation='softmax', name='token_predictions')(x)

# Создаем модель
model = keras.Model(inputs=inputs, outputs=outputs, name='decoder_only_transformer')

# Компилируем
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=masked_sparse_crossentropy,
    metrics=[masked_token_accuracy]
)

print("Модель собрана:")
model.summary()

# Подготовка данных
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.batch(cfg['batch_size']).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_dataset = val_dataset.batch(cfg['batch_size']).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_dataset = test_dataset.batch(cfg['batch_size']).prefetch(tf.data.AUTOTUNE)

# Функция для генерации с teacher forcing
def compute_generation_metrics(model, val_window_starts, val_X, val_y, char_to_id, id_to_char, max_new_tokens=CHECK_GEN_STEPS):
    """Вычисляет метрики генерации с teacher forcing."""
    successes = 0
    match_ratios = []
    
    # Берем первые PROMPT_COUNT окон из валидации
    for i in range(min(PROMPT_COUNT, len(val_X))):
        context = val_X[i]
        true_targets = val_y[i]
        
        # Используем teacher forcing: предсказываем следующий токен на основе истинной истории
        correct_predictions = 0
        
        for pos in range(max_new_tokens):
            # Берем контекст до текущей позиции
            current_context = context[:CONTEXT_LEN - max_new_tokens + pos + 1]
            if len(current_context) < 1:
                break
                
            # Предсказываем следующий токен
            input_tensor = tf.expand_dims(current_context, axis=0)
            pred_dist = model(input_tensor, training=False)
            pred_token = tf.argmax(pred_dist[0, -1, :]).numpy()
            
            # Сравниваем с истинным следующим токеном
            true_token = true_targets[pos] if pos < len(true_targets) else PAD_ID
            
            if pred_token == true_token:
                correct_predictions += 1
        
        match_ratio = correct_predictions / max_new_tokens
        match_ratios.append(match_ratio)
        
        if match_ratio >= cfg['gen_mean_threshold']:
            successes += 1
    
    return successes, np.mean(match_ratios)


# TODO 4.3: Реализуйте отдельный warm-up этап (JIT/компиляция)
print("\n" + "="*70)
print("ФАЗА 1: WARM-UP (компиляция, не входит в бюджет)")
print("="*70)

warmup_start = time.time()

# Небольшой warm-up батч
warmup_batch_X = X_train[:cfg['batch_size']]
warmup_batch_y = y_train[:cfg['batch_size']]

# Прогоняем warmup_steps итераций для компиляции
for step in range(cfg['warmup_steps']):
    _ = model.train_on_batch(warmup_batch_X, warmup_batch_y)
    if step == 0:
        print(f"  Warm-up итерация {step+1}: компиляция JIT...")
    else:
        print(f"  Warm-up итерация {step+1}: стабилизация...")

warmup_minutes = (time.time() - warmup_start) / 60
print(f"\nWarm-up завершен за {warmup_minutes:.2f} минут")

# TODO 4.4: Запустите измеряемый цикл обучения с лимитом времени
print("\n" + "="*70)
print(f"ФАЗА 2: ИЗМЕРЯЕМОЕ ОБУЧЕНИЕ (бюджет: {cfg['max_training_minutes']} минут)")
print("="*70)

timed_start = time.time()
training_trace = []
step = 0
best_val_loss = float('inf')
early_stop_counter = 0
generation_stop_counter = 0
stop_reason = None
generation_goal_achieved = False

# Паттерны для generation-probe
probe_success_history = []
probe_timestamps = []

# Основной цикл обучения
while True:
    epoch_start = time.time()
    
    for batch_x, batch_y in train_dataset:
        step += 1
        
        # Обучаем один шаг
        train_loss, train_acc = model.train_on_batch(batch_x, batch_y)
        
        # TODO 4.5: Проверка на валидации каждые eval_every_steps
        if step % cfg['eval_every_steps'] == 0:
            eval_start = time.time()
            
            # Валидация
            val_losses = []
            val_accs = []
            for val_x, val_y in val_dataset:
                v_loss, v_acc = model.test_on_batch(val_x, val_y)
                val_losses.append(v_loss)
                val_accs.append(v_acc)
            
            val_loss = np.mean(val_losses)
            val_acc = np.mean(val_accs)
            eval_seconds = time.time() - eval_start
            
            # TODO 4.6: Generation-probe на валидации
            probe_start = time.time()
            probe_success_count, probe_mean_match_ratio = compute_generation_metrics(
                model, starts_val, X_val, y_val, char_to_id, id_to_char
            )
            probe_seconds = time.time() - probe_start
            
            # Время
            round_train_seconds = (time.time() - epoch_start) - eval_seconds - probe_seconds
            round_total_seconds = time.time() - epoch_start
            elapsed_minutes = (time.time() - timed_start) / 60
            
            # Сохраняем trace
            training_trace.append({
                'step': step,
                'train_loss': float(train_loss),
                'val_loss': float(val_loss),
                'val_token_accuracy': float(val_acc),
                'probe_success_count': probe_success_count,
                'probe_mean_match_ratio': probe_mean_match_ratio,
                'round_train_seconds': round_train_seconds,
                'round_eval_seconds': eval_seconds,
                'round_probe_seconds': probe_seconds,
                'round_total_seconds': round_total_seconds,
                'stop_reason': None,
                'elapsed_minutes': elapsed_minutes,
                'completed_steps': step,
                'warmup_minutes': warmup_minutes,
                'budget_minutes': cfg['max_training_minutes']
            })
            
            # Вывод прогресса
            print(f"Step {step:6d} | Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | "
                  f"Probe: {probe_success_count}/{PROMPT_COUNT} ({probe_mean_match_ratio:.2%}) | "
                  f"Time: {elapsed_minutes:.1f}/{cfg['max_training_minutes']} min")
            
            # TODO 4.7: Проверка достижения цели генерации
            generation_goal_achieved = probe_success_count >= cfg['gen_threshold']
            
            if generation_goal_achieved:
                probe_success_history.append(True)
                probe_timestamps.append(elapsed_minutes)
                print(f"  ✓ Цель генерации достигнута! ({probe_success_count}/{cfg['gen_threshold']})")
                
                # TODO 4.9: Стабильное достижение цели
                if len(probe_success_history) >= cfg['generation_stop_patience']:
                    consecutive_successes = sum(probe_success_history[-cfg['generation_stop_patience']:])
                    
                    # Проверяем минимальное время
                    min_time_ok = elapsed_minutes >= cfg['min_minutes_before_generation_stop']
                    
                    if consecutive_successes >= cfg['generation_stop_patience'] and min_time_ok:
                        stop_reason = 'generation_goal_stable'
                        print(f"\n🎯 Остановка: стабильное достижение цели генерации "
                              f"({consecutive_successes} проверок подряд, {elapsed_minutes:.1f} мин)")
                        break
            else:
                probe_success_history.append(False)
                probe_timestamps.append(elapsed_minutes)
                generation_stop_counter = 0
            
            # TODO 4.8: Ранняя остановка по val_loss (вспомогательная)
            if val_loss < best_val_loss - cfg['early_stopping_min_delta']:
                best_val_loss = val_loss
                early_stop_counter = 0
            else:
                early_stop_counter += 1
            
            # Применяем early stopping только если достигнуто минимальное время
            elapsed_minutes_check = (time.time() - timed_start) / 60
            if (early_stop_counter >= cfg['early_stopping_patience'] and 
                elapsed_minutes_check >= cfg['min_minutes_before_early_stop'] and
                not generation_goal_achieved):
                stop_reason = 'early_stopping_val_loss'
                print(f"\n⏹️ Ранняя остановка по val_loss (пациенс: {early_stop_counter})")
                break
            
            epoch_start = time.time()
    
    # Проверяем выход из внутреннего цикла
    if stop_reason is not None:
        break
    
    # TODO 4.4: Проверка временного бюджета
    elapsed_minutes = (time.time() - timed_start) / 60
    if elapsed_minutes >= cfg['max_training_minutes']:
        stop_reason = 'time_budget_exceeded'
        print(f"\n⏰ Остановка: достигнут лимит времени ({elapsed_minutes:.1f}/{cfg['max_training_minutes']} мин)")
        break

# Обновляем stop_reason в trace
for trace in training_trace:
    if trace['stop_reason'] is None:
        trace['stop_reason'] = stop_reason

final_elapsed = (time.time() - timed_start) / 60
print(f"\nОбучение завершено. Причина: {stop_reason}")
print(f"Итоговое время: {final_elapsed:.2f} минут")
print(f"Шагов выполнено: {step}")

# TODO 4.10: Формируем training_trace (уже сделано выше)

# TODO 4.11: Финальная оценка на тестовой выборке
print("\n" + "="*70)
print("ФИНАЛЬНАЯ ОЦЕНКА НА ТЕСТОВОЙ ВЫБОРКЕ")
print("="*70)

test_losses = []
test_accs = []
for test_x, test_y in test_dataset:
    t_loss, t_acc = model.test_on_batch(test_x, test_y)
    test_losses.append(t_loss)
    test_accs.append(t_acc)

test_loss = np.mean(test_losses)
test_accuracy = np.mean(test_accs)
test_perplexity = perplexity_from_loss(test_loss)

# Вычисляем baseline перплексию
baseline_perp = frequency_baseline_perplexity(y_train, y_test, VOCAB_SIZE, PAD_ID)

print(f"\nРезультаты на тесте:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_accuracy:.4f}")
print(f"  Test Perplexity: {test_perplexity:.2f}")
print(f"  Baseline Perplexity: {baseline_perp:.2f}")
print(f"  CPU Reference Perplexity: {CPU_REFERENCE_PERPLEXITY:.2f}")

# Проверка критериев
success_count, mean_match_ratio = compute_generation_metrics(
    model, starts_val, X_val, y_val, char_to_id, id_to_char
)

print(f"\nКритерии успеха:")
print(f"  Generation success: {success_count}/{cfg['gen_threshold']} (need >= {cfg['gen_threshold']})")
print(f"  Mean match ratio: {mean_match_ratio:.2%} (need >= {cfg['gen_mean_threshold']:.2%})")
print(f"  Test Perplexity < Baseline: {test_perplexity:.2f} < {baseline_perp:.2f} = {test_perplexity < baseline_perp}")
print(f"  Test Perplexity < CPU Reference: {test_perplexity:.2f} < {CPU_REFERENCE_PERPLEXITY:.2f} = {test_perplexity < CPU_REFERENCE_PERPLEXITY}")

# Итоговый вердикт
overall_pass = (
    success_count >= cfg['gen_threshold'] and
    mean_match_ratio >= cfg['gen_mean_threshold'] and
    test_perplexity < baseline_perp
)

print(f"\n{'='*70}")
print(f"ИТОГ: {'✅ ПРОЙДЕНО' if overall_pass else '❌ НЕ ПРОЙДЕНО'}")
print(f"{'='*70}")

In [ ]:
# Графики и контроль после TODO 4
steps = training_trace['step']
train_losses = training_trace['train_loss']
val_losses = training_trace['val_loss']

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(steps, train_losses, label='train_loss')
plt.plot(steps, val_losses, label='val_loss')
plt.xlabel('шаг обучения')
plt.ylabel('loss')
plt.legend()

plt.subplot(1, 2, 2)
train_ppl = [perplexity_from_loss(v) for v in train_losses]
val_ppl = [perplexity_from_loss(v) for v in val_losses]
plt.plot(steps, train_ppl, label='train_ppl')
plt.plot(steps, val_ppl, label='val_ppl')
plt.xlabel('шаг обучения')
plt.ylabel('perplexity')
plt.legend()
plt.tight_layout()

print(f"Причина остановки    : {training_trace['stop_reason']}")
print(f"Warm-up (мин)        : {training_trace['warmup_minutes']:.2f}")
print(f"Бюджет после warm-up : {training_trace['budget_minutes']:.2f}")
print(f"Измеряемое время (мин): {training_trace['elapsed_minutes']:.2f}")
print(f"Выполнено шагов      : {training_trace['completed_steps']}")
print(f"test_loss            : {test_loss:.4f}")
print(f"test_token_accuracy  : {test_token_accuracy:.4f}")
print(f"test_perplexity      : {test_perplexity:.4f}")
print(f"baseline_perplexity  : {baseline_perplexity:.4f}")
print(f"baseline_pass        : {baseline_pass}")
print(f"cpu_reference_pass   : {cpu_reference_pass}")


## TODO 5: детерминированная генерация по фиксированным подсказкам


In [ ]:
def ids_to_text(token_ids, id_to_char_map):
    """Преобразует идентификаторы в строку.

    Аргументы:
      token_ids: Последовательность идентификаторов.
      id_to_char_map: Отображение id -> символ.

    Возвращает:
      Строка символов.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return ''.join(id_to_char_map.get(int(token), '') for token in token_ids if int(token) != PAD_ID)


def generate_greedy(model, prompt_ids, steps, context_len):
    """Генерирует продолжение в режиме argmax через свободную автогенерацию.

    Аргументы:
      model: Обученная модель.
      prompt_ids: Начальная последовательность идентификаторов.
      steps: Число новых токенов.
      context_len: Длина модельного контекста.

    Возвращает:
      Список токенов продолжения длины не больше `steps`.

    Исключения:
      ValueError: Если подсказка пуста.
    """
    if len(prompt_ids) == 0:
        raise ValueError('Подсказка не может быть пустой.')
    
    # Копируем начальную последовательность
    generated = list(prompt_ids)
    
    for _ in range(steps):
        # Берем последние context_len токенов
        current_seq = generated[-context_len:] if len(generated) > context_len else generated
        
        # Преобразуем в тензор
        input_tensor = tf.expand_dims(tf.convert_to_tensor(current_seq), axis=0)
        
        # Получаем предсказания (без training=False для инференса)
        predictions = model(input_tensor, training=False)
        
        # Берем последний токен и выбираем argmax
        next_token_logits = predictions[0, -1, :]
        next_token = tf.argmax(next_token_logits).numpy()
        
        # Добавляем в последовательность
        generated.append(next_token)
    
    # Возвращаем только сгенерированные токены (без промпта)
    return generated[len(prompt_ids):]


def build_prompt_targets(encoded_stream, start_indices, context_len, continuation_len, n_prompts):
    """Готовит фиксированные подсказки и эталонные продолжения из тестового фрагмента.

    Аргументы:
      encoded_stream: Полный поток кодов корпуса.
      start_indices: Стартовые индексы окон тестовой части.
      context_len: Длина контекста модели.
      continuation_len: Длина целевого продолжения.
      n_prompts: Количество подсказок.

    Возвращает:
      Список пар `(prompt_ids, target_ids)`.

    Исключения:
      ValueError: Если тестовых окон недостаточно.
    """
    valid_starts = [
        int(start) for start in start_indices
        if int(start) + context_len + continuation_len <= len(encoded_stream)
    ]
    if len(valid_starts) < n_prompts:
        raise ValueError('Недостаточно тестовых окон для фиксированного набора подсказок.')

    selected = np.linspace(0, len(valid_starts) - 1, n_prompts, dtype=int)
    pairs = []
    for idx in selected:
        start = valid_starts[int(idx)]
        prompt = encoded_stream[start:start + context_len]
        target = encoded_stream[start + context_len:start + context_len + continuation_len]
        pairs.append((prompt.tolist(), target.tolist()))
    return pairs


# TODO 5.2: Посчитайте success_count и mean_match_ratio по контролируемому
# продолжению (teacher forcing) на фиксированных test-подсказках.

print("\n" + "="*70)
print("ОЦЕНКА КОНТРОЛИРУЕМОГО ПРОДОЛЖЕНИЯ (TEACHER FORCING)")
print("="*70)

# Параметры для оценки
CONTINUATION_LEN = CHECK_GEN_STEPS  # 16 токенов
N_PROMPTS = PROMPT_COUNT  # 20 подсказок

# Строим пары промпт-цель из тестовой выборки
test_pairs = build_prompt_targets(
    encoded_ids, 
    starts_test, 
    CONTEXT_LEN, 
    CONTINUATION_LEN, 
    N_PROMPTS
)

print(f"Сформировано {len(test_pairs)} тестовых пар")
print(f"Длина контекста: {CONTEXT_LEN}, длина продолжения: {CONTINUATION_LEN}")

# Оценка с teacher forcing
teacher_forcing_successes = 0
teacher_forcing_match_ratios = []

for i, (prompt, target) in enumerate(test_pairs):
    correct_predictions = 0
    
    # Для каждого шага используем истинную историю
    for step in range(CONTINUATION_LEN):
        # Берем контекст: промпт + истинные токены до текущей позиции
        if step == 0:
            current_context = prompt
        else:
            current_context = prompt + target[:step]
        
        # Убеждаемся, что контекст не длиннее CONTEXT_LEN
        if len(current_context) > CONTEXT_LEN:
            current_context = current_context[-CONTEXT_LEN:]
        
        # Предсказываем следующий токен
        input_tensor = tf.expand_dims(tf.convert_to_tensor(current_context), axis=0)
        predictions = model(input_tensor, training=False)
        pred_token = tf.argmax(predictions[0, -1, :]).numpy()
        
        # Сравниваем с истинным целевым токеном
        true_token = target[step]
        if pred_token == true_token:
            correct_predictions += 1
    
    match_ratio = correct_predictions / CONTINUATION_LEN
    teacher_forcing_match_ratios.append(match_ratio)
    
    if match_ratio >= cfg['gen_mean_threshold']:
        teacher_forcing_successes += 1
        status = "✓"
    else:
        status = "✗"
    
    if i < 5 or i >= N_PROMPTS - 3:  # Показываем первые 5 и последние 3
        print(f"  {status} Промпт {i+1:2d}: совпадений {correct_predictions}/{CONTINUATION_LEN} "
              f"({match_ratio:.2%})")
    elif i == 5:
        print("  ...")

print(f"\nTeacher Forcing результаты:")
print(f"  Успешных промптов: {teacher_forcing_successes}/{N_PROMPTS} "
      f"(нужно >= {cfg['gen_threshold']})")
print(f"  Средняя доля совпадений: {np.mean(teacher_forcing_match_ratios):.2%} "
      f"(нужно >= {cfg['gen_mean_threshold']:.2%})")

teacher_forcing_pass = (teacher_forcing_successes >= cfg['gen_threshold'] and 
                        np.mean(teacher_forcing_match_ratios) >= cfg['gen_mean_threshold'])

# TODO 5.3: Свободную автогенерацию оставьте как демонстрационный блок примеров.

print("\n" + "="*70)
print("ДЕМОНСТРАЦИЯ СВОБОДНОЙ АВТОГЕНЕРАЦИИ")
print("="*70)

free_generation_examples = []
for i in range(min(5, len(test_pairs))):
    prompt, target = test_pairs[i]
    
    # Свободная генерация (без teacher forcing)
    generated_tokens = generate_greedy(model, prompt, CONTINUATION_LEN, CONTEXT_LEN)
    
    # Преобразуем в текст
    prompt_text = ids_to_text(prompt, id_to_char)
    target_text = ids_to_text(target, id_to_char)
    generated_text = ids_to_text(generated_tokens, id_to_char)
    
    free_generation_examples.append({
        'prompt': prompt_text,
        'target': target_text,
        'generated': generated_text,
        'match_ratio': sum(1 for a,b in zip(generated_tokens, target) if a==b) / len(target)
    })
    
    print(f"\nПример {i+1}:")
    print(f"  Промпт:          \"{prompt_text[:80]}...\"")
    print(f"  Ожидалось:       \"{target_text}\"")
    print(f"  Сгенерировано:   \"{generated_text}\"")
    print(f"  Совпадение:      {free_generation_examples[-1]['match_ratio']:.2%}")

# TODO 5.4: Соберите run_summary и выведите строку JSON

print("\n" + "="*70)
print("ФОРМИРОВАНИЕ ИТОГОВОГО ОТЧЕТА")
print("="*70)

# Вычисляем финальные метрики на тесте
final_test_losses = []
final_test_accs = []
for test_x, test_y in test_dataset:
    t_loss, t_acc = model.test_on_batch(test_x, test_y)
    final_test_losses.append(t_loss)
    final_test_accs.append(t_acc)

final_test_loss = np.mean(final_test_losses)
final_test_accuracy = np.mean(final_test_accs)
final_test_perplexity = perplexity_from_loss(final_test_loss)

# Baseline перплексия
baseline_perp = frequency_baseline_perplexity(y_train, y_test, VOCAB_SIZE, PAD_ID)

# Проверка критериев
baseline_pass = final_test_perplexity < baseline_perp
generation_pass = teacher_forcing_successes >= cfg['gen_threshold']
cpu_reference_pass = final_test_perplexity < CPU_REFERENCE_PERPLEXITY
overall_pass = baseline_pass and generation_pass

# Формируем run_summary
run_summary = {
    'status': 'PASS' if overall_pass else 'FAIL',
    'overall_pass': overall_pass,
    'test_perplexity': round(float(final_test_perplexity), 4),
    'test_accuracy': round(float(final_test_accuracy), 4),
    'baseline_perplexity': round(float(baseline_perp), 4),
    'baseline_pass': baseline_pass,
    'cpu_reference_perplexity': CPU_REFERENCE_PERPLEXITY,
    'cpu_reference_pass': cpu_reference_pass,
    'teacher_forcing': {
        'success_count': teacher_forcing_successes,
        'threshold': cfg['gen_threshold'],
        'generation_pass': generation_pass,
        'mean_match_ratio': round(float(np.mean(teacher_forcing_match_ratios)), 4),
        'mean_threshold': cfg['gen_mean_threshold']
    },
    'training': {
        'stop_reason': stop_reason,
        'total_steps': step,
        'elapsed_minutes': round(final_elapsed, 2),
        'budget_minutes': cfg['max_training_minutes'],
        'warmup_minutes': round(warmup_minutes, 2),
        'final_val_loss': round(float(training_trace[-1]['val_loss']), 4) if training_trace else None,
        'final_val_accuracy': round(float(training_trace[-1]['val_token_accuracy']), 4) if training_trace else None
    },
    'model_config': {
        'context_len': CONTEXT_LEN,
        'embed_dim': EMBED_DIM,
        'num_heads': NUM_HEADS,
        'ff_dim': FF_DIM,
        'num_layers': NUM_LAYERS,
        'vocab_size': VOCAB_SIZE
    },
    'profile': selected_profile_name
}

# Выводим JSON строку для парсинга скриптом
import json
run_summary_json = json.dumps(run_summary, indent=2)
print("\n" + "="*70)
print("GPU_RUN_SUMMARY_JSON=")
print(run_summary_json)
print("="*70)

# TODO 5.5: Сравнение с CPU_REFERENCE_PERPLEXITY оставьте индикатором (без assert)

print("\n" + "="*70)
print("ИТОГОВЫЙ ВЕРДИКТ")
print("="*70)
print(f"  ✅ Baseline пройден: {test_perplexity:.2f} < {baseline_perp:.2f}" if baseline_pass else f"  ❌ Baseline не пройден: {test_perplexity:.2f} >= {baseline_perp:.2f}")
print(f"  ✅ Generation пройден: {teacher_forcing_successes} >= {cfg['gen_threshold']}" if generation_pass else f"  ❌ Generation не пройден: {teacher_forcing_successes} < {cfg['gen_threshold']}")
print(f"  📊 CPU Reference: {test_perplexity:.2f} {'<' if cpu_reference_pass else '>'} {CPU_REFERENCE_PERPLEXITY:.2f} (индикатор)")

print(f"\n{'='*70}")
print(f"РЕЗУЛЬТАТ: {'✅ ПРОЙДЕНО' if overall_pass else '❌ НЕ ПРОЙДЕНО'}")
print(f"{'='*70}")

if overall_pass:
    print("\nПоздравляем! GPU лабораторная работа успешно выполнена!")
    print(f"Модель достигла {teacher_forcing_successes}/{cfg['gen_threshold']} успешных продолжений")
    print(f"с перплексией {test_perplexity:.2f} (baseline: {baseline_perp:.2f})")
else:
    print("\nРекомендации по улучшению:")
    if not generation_pass:
        print("  - Увеличьте размер модели (embed_dim, num_layers)")
        print("  - Попробуйте profile 'gpu_60m_boost' с измененным learning_rate")
        print("  - Увеличьте бюджет времени через GPU_TRAINING_BUDGET_MINUTES")
    if not baseline_pass:
        print("  - Модель не превзошла частотный baseline, требуется больше обучения")

## TODO 6: диагностика внимания


In [ ]:
# TODO 6.1: Постройте trace-модель, которая возвращает attention_scores из decoder_layer.
# TODO 6.2: Усредните веса внимания по головам и вычислите отношение массы в будущем.
# TODO 6.3: Проверьте, что отношение массы в будущем меньше 1e-4.

print("\n" + "="*70)
print("ДИАГНОСТИКА ВНИМАНИЯ: ПРОВЕРКА CAUSAL MASK")
print("="*70)

# Функция для создания модели, возвращающей attention scores
def build_attention_tracing_model(base_model, decoder_block_index=-1):
    """Создает модель, которая возвращает attention_scores из указанного блока."""
    # Находим все CausalDecoderBlock слои
    decoder_blocks = [layer for layer in base_model.layers if isinstance(layer, CausalDecoderBlock)]
    
    if not decoder_blocks:
        raise ValueError("В модели нет CausalDecoderBlock слоев")
    
    # Берем последний блок по умолчанию
    target_block = decoder_blocks[decoder_block_index]
    
    # Строим граф для получения attention scores
    input_layer = base_model.input
    x = input_layer
    
    # Проходим через все слои до целевого блока
    attention_scores = None
    for layer in base_model.layers:
        if layer == target_block:
            # Для целевого блока вызываем с return_attention_scores=True
            x, attention_scores = layer(x, return_attention_scores=True)
        elif isinstance(layer, CausalDecoderBlock):
            # Для остальных декодерных блоков без attention scores
            x = layer(x)
        elif layer.name != base_model.layers[0].name:  # Пропускаем input слой
            x = layer(x)
    
    if attention_scores is None:
        raise ValueError("Не удалось получить attention scores")
    
    tracing_model = keras.Model(inputs=input_layer, outputs=attention_scores, name='attention_tracer')
    return tracing_model

# Строим trace-модель
try:
    attention_tracer = build_attention_tracing_model(model, decoder_block_index=-1)
    print("✓ Trace-модель для диагностики внимания создана")
    print(f"  Выходная форма: attention_scores (batch, num_heads, query_len, key_len)")
except Exception as e:
    print(f"✗ Ошибка при создании trace-модели: {e}")
    raise

# Берем несколько примеров из тестовой выборки для диагностики
test_indices_for_attention = [0, 1, 2, 5, 10]  # Фиксированные индексы
attention_ratios = []

for idx in test_indices_for_attention:
    if idx >= len(X_test):
        continue
    
    sample_input = X_test[idx:idx+1]  # (1, context_len)
    
    # Получаем attention scores
    attention_scores = attention_tracer(sample_input, training=False)
    print(f"\nПример {idx}: attention_scores shape = {attention_scores.shape}")
    
    # TODO 6.2: Усредните веса внимания по головам
    # attention_scores: (batch, num_heads, query_len, key_len)
    avg_attention = tf.reduce_mean(attention_scores, axis=1)  # (batch, query_len, key_len)
    avg_attention = avg_attention[0].numpy()  # Убираем batch dimension
    
    # Находим непустые токены во входе
    input_tokens = X_test[idx]
    non_pad_mask = input_tokens != PAD_ID
    non_pad_indices = np.where(non_pad_mask)[0]
    non_pad_len = len(non_pad_indices)
    
    if non_pad_len < 2:
        print(f"  Пропуск: слишком короткая последовательность ({non_pad_len} токенов)")
        continue
    
    # Обрезаем attention карту до непустых позиций
    trimmed_attention = avg_attention[:non_pad_len, :non_pad_len]
    
    # Вычисляем массу внимания к будущим токенам (выше диагонали)
    # Создаем маску для будущих токенов (верхний треугольник, исключая диагональ)
    future_mask = np.triu(np.ones_like(trimmed_attention), k=1).astype(bool)
    past_mask = np.tril(np.ones_like(trimmed_attention), k=0).astype(bool)
    
    future_attention_mass = trimmed_attention[future_mask].sum()
    past_attention_mass = trimmed_attention[past_mask].sum()
    total_attention_mass = trimmed_attention.sum()
    
    # Отношение массы внимания к будущему
    future_ratio = future_attention_mass / total_attention_mass if total_attention_mass > 0 else 0
    
    attention_ratios.append(future_ratio)
    
    print(f"  Длина последовательности (непустые): {non_pad_len}")
    print(f"  Масса внимания к прошлому+текущему: {past_attention_mass:.6f}")
    print(f"  Масса внимания к будущему: {future_attention_mass:.6f}")
    print(f"  Отношение (будущее/все): {future_ratio:.2e}")
    
    # Визуализация attention карты для первого примера
    if idx == test_indices_for_attention[0]:
        plt.figure(figsize=(10, 8))
        
        # Ограничим размер для визуализации (максимум 50 токенов)
        vis_len = min(non_pad_len, 50)
        vis_attention = trimmed_attention[:vis_len, :vis_len]
        
        plt.imshow(vis_attention, cmap='Blues', aspect='auto')
        plt.colorbar(label='Attention weight')
        plt.xlabel('Key positions (context)')
        plt.ylabel('Query positions (current)')
        plt.title(f'Attention Map (averaged over heads)\nFirst {vis_len} non-pad tokens')
        
        # Рисуем красную линию по диагонали (causal boundary)
        plt.plot([-0.5, vis_len-0.5], [-0.5, vis_len-0.5], 'r--', linewidth=2, 
                 label='causal boundary (diagonal)')
        plt.legend()
        plt.tight_layout()
        plt.show()
        
        # Дополнительная визуализация: heatmap разницы
        plt.figure(figsize=(10, 8))
        # Создаем маску нарушения причинности (будущие токены)
 violation_mask = np.triu(np.ones_like(vis_attention), k=1)
        violation_weights = vis_attention * violation_mask
        
        plt.imshow(violation_weights, cmap='Reds', aspect='auto', vmin=0, vmax=vis_attention.max())
        plt.colorbar(label='Illegal attention weight')
        plt.xlabel('Key positions (context)')
        plt.ylabel('Query positions (current)')
        plt.title('Illegal Future Attention (should be all zeros)')
        plt.tight_layout()
        plt.show()

# TODO 6.3: Проверьте, что отношение массы внимания в будущем меньше 1e-4.
print("\n" + "="*70)
print("ПРОВЕРКА CAUSAL MASK")
print("="*70)

threshold = 1e-4
all_passed = True

for i, ratio in enumerate(attention_ratios):
    status = "✓" if ratio < threshold else "✗"
    print(f"  {status} Пример {test_indices_for_attention[i]}: {ratio:.2e} < {threshold} = {ratio < threshold}")
    if ratio >= threshold:
        all_passed = False

# Статистика по всем проверенным примерам
mean_ratio = np.mean(attention_ratios)
max_ratio = np.max(attention_ratios)
min_ratio = np.min(attention_ratios)

print(f"\nСтатистика по {len(attention_ratios)} примерам:")
print(f"  Среднее отношение: {mean_ratio:.2e}")
print(f"  Максимальное: {max_ratio:.2e}")
print(f"  Минимальное: {min_ratio:.2e}")

# Проверка на дополнительных случайных примерах для надежности
print("\nДополнительная проверка на 20 случайных примерах...")
random_indices = np.random.choice(len(X_test), min(20, len(X_test)), replace=False)
random_ratios = []

for idx in random_indices:
    sample_input = X_test[idx:idx+1]
    attention_scores = attention_tracer(sample_input, training=False)
    avg_attention = tf.reduce_mean(attention_scores, axis=1)[0].numpy()
    
    input_tokens = X_test[idx]
    non_pad_len = np.sum(input_tokens != PAD_ID)
    
    if non_pad_len >= 2:
        trimmed = avg_attention[:non_pad_len, :non_pad_len]
        future_mask = np.triu(np.ones_like(trimmed), k=1).astype(bool)
        future_mass = trimmed[future_mask].sum()
        total_mass = trimmed.sum()
        ratio = future_mass / total_mass if total_mass > 0 else 0
        random_ratios.append(ratio)

random_mean = np.mean(random_ratios) if random_ratios else 0
random_max = np.max(random_ratios) if random_ratios else 0

print(f"  Случайная выборка ({len(random_ratios)} примеров):")
print(f"    Среднее отношение: {random_mean:.2e}")
print(f"    Максимальное: {random_max:.2e}")

# Итоговый вердикт
print("\n" + "="*70)
print("ИТОГ ДИАГНОСТИКИ ВНИМАНИЯ")
print("="*70)

if all_passed and random_max < threshold:
    print("✅ CAUSAL MASK РАБОТАЕТ КОРРЕКТНО")
    print(f"   Все проверенные примеры имеют отношение внимания к будущему < {threshold}")
    print(f"   Среднее отношение: {mean_ratio:.2e}")
    print("   Модель не имеет доступа к будущим токенам")
else:
    print("❌ ОБНАРУЖЕНА УТЕЧКА ИНФОРМАЦИИ")
    print(f"   Найдены примеры с отношением >= {threshold}")
    if not all_passed:
        print("   Проверьте реализацию causal mask в CausalDecoderBlock")
    if random_max >= threshold:
        print("   Проблема воспроизводится на случайных примерах")

# Добавляем результаты диагностики в run_summary
run_summary['attention_diagnostics'] = {
    'passed': all_passed and random_max < threshold,
    'mean_future_ratio': float(mean_ratio),
    'max_future_ratio': float(max_ratio),
    'random_mean_ratio': float(random_mean),
    'random_max_ratio': float(random_max),
    'threshold': threshold,
    'samples_checked': len(attention_ratios) + len(random_ratios)
}

# Обновляем overall_pass с учетом диагностики внимания
run_summary['attention_pass'] = all_passed and random_max < threshold
run_summary['overall_pass'] = run_summary['overall_pass'] and run_summary['attention_pass']

print("\n" + "="*70)
print("ОКОНЧАТЕЛЬНЫЙ ВЕРДИКТ")
print("="*70)
print(f"  Baseline пройден: {run_summary['baseline_pass']}")
print(f"  Generation пройден: {run_summary['teacher_forcing']['generation_pass']}")
print(f"  Attention диагностика: {run_summary['attention_pass']}")
print(f"\n  ИТОГ: {'✅ ПРОЙДЕНО' if run_summary['overall_pass'] else '❌ НЕ ПРОЙДЕНО'}")

# Выводим обновленный JSON
print("\n" + "="*70)
print("ОБНОВЛЕННЫЙ GPU_RUN_SUMMARY_JSON:")
print(json.dumps(run_summary, indent=2))
print("="*70)

## Чек-лист перед завершением `ЛР02` (GPU-вариант)

1. Все `TODO` закрыты.
2. `gpu_preflight()` пройден полностью.
3. Выполнен отдельный warm-up; его время не входит в измеряемый бюджет.
4. Измеряемый бюджет после warm-up: `60` минут (`cfg['max_training_minutes']`).
5. Жёсткий критерий `19/20` считается по контролируемому продолжению (teacher forcing).
6. Дополнительно достигнут средний порог `mean_match_ratio` для контролируемого продолжения.
7. `test_perplexity < baseline_perplexity`.
8. `test_perplexity < CPU_REFERENCE_PERPLEXITY` — индикаторный контроль (без аварийного assert).
9. Свободная автогенерация остаётся демонстрационным блоком.
10. Диагностика внимания подтверждает отсутствие доступа в будущее.
